# TRADING INTELLIGENT · CH03
# Laboratoire intégral — Comprendre et prévoir le risque

**Économétrie financière · Mathématiques appliquées · Intelligence artificielle**  
**Tawfik Masrour — Département d’informatique, Faculté des sciences, Université Mohammed V de Rabat**

*Édition de laboratoire 1.0 · 2 novembre 2026*

> **Notre promesse pédagogique**  
> Nous n'utiliserons pas une fonction Python pour éviter de comprendre un concept. Chaque grande idée naît d'une question, se construit avec quelques nombres, puis devient une expérience reproductible. **L'ordinateur vérifie le raisonnement : il ne le remplace pas.**

Dans la vie d'un analyste financier, on connaît rarement la direction du prochain rendement. Mais une chose peut évoluer de manière plus structurée : **l'ampleur possible des fluctuations**. Tout ce laboratoire part de cette distinction.

Ce notebook appartient au cours **CH03 — Comprendre et prévoir le risque**. Il est conçu pour être lu **de haut en bas**, seul, sans connaissance préalable approfondie des probabilités ni de l'économétrie. Des connaissances élémentaires d'algèbre sont suffisantes.

**Statut des données :** jeu de données entièrement **synthétique**, avec paramètres générateurs connus, uniquement pour enseigner les mécanismes. Les résultats numériques **ne constituent pas un signal de trading**, une recommandation financière, ni la validation empirique d'une stratégie réelle.

## Comment apprendre avec ce laboratoire ?

Nous répéterons un même mouvement :

1. **Voir** une situation ou quelques chiffres.
2. **Se poser une question** qui justifie une nouvelle notion.
3. **Calculer à la main** un exemple minimal.
4. **Écrire et exécuter** quelques lignes de Python expliquées.
5. **Lire le résultat** : que montrent exactement les nombres et les axes ?
6. **Mettre à l'épreuve** l'idée : qu'arrive-t-il si un paramètre change, si le modèle est mal spécifié, ou si les données changent de régime ?
7. **Faire le lien** avec le chapitre et la décision financière.

La version **STUDENT** est exécutable intégralement, sans corrigés réservés à l'enseignant. Les questions ouvertes restent des invitations à interpréter ; le code fournit un point de départ opérationnel. La version **MASTER** ajoute des réponses argumentées. Dans les deux cas, les données sont créées par le notebook : **aucun téléchargement et aucune connexion réseau ne sont requis**.

### Table des parcours

| Étape | Notions du CH03 | Questions que l'on saura résoudre |
|:--|:--|:--|
| I | Prix, rendements, prévision, surprise | Que calcule-t-on réellement ? |
| II | Scénarios, information, espérance conditionnelle | Comment une information change-t-elle une prévision ? |
| III | Dispersion, variance, volatilité | Pourquoi une même moyenne ne signifie-t-elle pas un même risque ? |
| IV | Clustering, ACF, saisonnalité, ARCH-LM | Comment voir et tester la mémoire des amplitudes ? |
| V | ARCH, GARCH, persistance, prévisions | Comment le risque d'hier agit-il sur celui de demain ? |
| VI | Estimation, vraisemblance, QMLE | Comment les données choisissent-elles les paramètres ? |
| VII | Diagnostics, Student-t, GJR et EGARCH | Que manque-t-il au premier modèle ? |
| VIII | VaR, ES, horizons et pertes en DH | Comment traduire la volatilité en risque financier ? |
| IX | Proxys, QLIKE, Kupiec, Christoffersen | Une prévision est-elle vraiment crédible hors échantillon ? |
| X | Décision et étude de cas finale | Quel risque présenter et comment le défendre ? |

Les formules affichées sont cohérentes avec le cours : **les rendements de nos modèles seront souvent exprimés en pourcentage** (un rendement de `2` signifie `2 %`), et non en proportion décimale (`0.02`). Nous signalerons toujours l'unité.

## Environnement scientifique minimal

Nous utiliserons `numpy` pour les calculs numériques, `pandas` pour les tableaux datés, `matplotlib` pour les graphiques, `scipy` pour les distributions et l'optimisation et `statsmodels` pour quelques diagnostics comparatifs. **Aucune bibliothèque GARCH spécialisée n'est nécessaire : nous programmerons la récursion du modèle.**

Le générateur pseudo-aléatoire reçoit une graine fixe. Une même graine et les mêmes versions de bibliothèques permettent de retrouver les mêmes données ; les différences de plate-forme peuvent produire de légères variations numériques d'optimisation.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize, minimize_scalar
from scipy.special import expit, xlogy
from statsmodels.stats.diagnostic import het_arch

SEED = 20261102
np.set_printoptions(precision=4, suppress=True)
pd.set_option('display.max_columns', 12)
pd.set_option('display.width', 115)
plt.rcParams.update({'figure.figsize': (10, 3.8), 'figure.dpi': 110,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': .19})
COLOR = '#125b70'; ACCENT = '#d28b30'; MUTED = '#7c8b94'
print('Python',sys.version.split()[0], '| numpy',np.__version__,'| pandas',pd.__version__)
print('Environnement prêt ; aucune connexion réseau requise.')

### Comment lire nos figures ?

Chaque figure présente un **titre**, deux **axes** et leurs **unités**. Les courbes décrivent un phénomène dans un échantillon précis ; elles ne démontrent pas à elles seules une loi générale. Quand une expérience est **simulée**, nous le dirons. Lorsque nous afficherons un résultat de test, nous distinguerons clairement une suggestion statistique d'une certitude.

## Le marché-école, notre fil rouge

Nous avons besoin d'un terrain d'expérience qui ne « triche » pas. Nous fabriquerons donc un marché fictif sur environ 2 400 séances, où la **variance conditionnelle réelle est connue** du concepteur. Le mécanisme est construit pour présenter :

- des rendements de moyenne presque constante ;
- des phases de volatilité calme ou agitée ;
- des chocs négatifs pouvant agir différemment des positifs ;
- des innovations à queues plus épaisses qu'une normale.

La simulation suit une récursion **GJR-GARCH(1,1)** (qui sera expliquée plus loin) et une innovation Student-t **standardisée**. Le lecteur **n'a pas besoin de connaître ces mots maintenant** ; ils prendront sens, chapitre après chapitre. Il faut simplement retenir que nous connaissons ici la « vraie » variance cachée, chose impossible avec des rendements de marché ordinaires.

**Point d'éthique scientifique :** nous savons, par construction, qu'un modèle voisin du générateur peut mieux fonctionner. Ce succès n'est donc pas une démonstration de supériorité sur des données réelles.

In [ ]:
def fabriquer_marche(n=2400, seed=SEED):
    """Données synthétiques : rendements en pourcentage et variance en %²."""
    rng = np.random.default_rng(seed)
    nu = 7.0
    z = rng.standard_t(nu, size=n) * np.sqrt((nu-2)/nu)
    omega, alpha, gamma, beta = 0.015, 0.065, 0.085, 0.870
    mu = 0.025
    persistence = alpha + beta + gamma/2
    assert persistence < 1, 'La variance de long terme doit être finie dans cette simulation.'
    h = np.empty(n); eps = np.empty(n)
    h[0] = omega/(1-persistence)
    for t in range(n):
        if t > 0:
            neg = 1.0 if eps[t-1] < 0 else 0.0
            h[t] = omega + (alpha + gamma*neg)*eps[t-1]**2 + beta*h[t-1]
        eps[t] = np.sqrt(h[t])*z[t]
    dates = pd.bdate_range('2017-01-02', periods=n)
    r_pct = mu + eps
    p = 100.0*np.exp(np.cumsum(np.log1p(r_pct/100)))
    df = pd.DataFrame({'date':dates, 'prix_fictif':p, 'rendement_pct':r_pct,
                       'innovation_pct':eps, 'variance_latente_pct2':h,
                       'volatilite_latente_pct':np.sqrt(h), 'z_vrai':z})
    return df, {'omega':omega,'alpha':alpha,'gamma':gamma,'beta':beta,'nu':nu,'mu':mu}

marche, VRAIS_PARAMETRES = fabriquer_marche()
print('Premières lignes de notre marché entièrement simulé :')
display(marche.head(4).round(4))
print('Nombre de séances :',len(marche),'| moyenne simulée (arrondie) :',round(marche.rendement_pct.mean(),3),'%')

**Vocabulaire à conserver :** `prix_fictif` est une valeur artificielle calculée à partir des rendements simulés, pas le cours d'un actif réel. `rendement_pct` s'exprime en points de pourcentage. `variance_latente_pct2` est une **variance cachée vraie dans notre simulation**, et ne serait pas observable sur un marché réel.

L'équation qui détermine le prix fictif dans cette cellule sert uniquement à créer une série cohérente ; nous expliquerons à présent pourquoi un rendement est plus pratique qu'un prix pour étudier le risque.

---
## I · Prix, rendements, prévisions et surprises

*Dans le cours : §§ 1–3*  

> **La question qui ouvre cette séquence**  
> Pourquoi une variation de 10 DH ne signifie-t-elle pas la même chose pour deux prix différents ?

**Notre itinéraire :** prix → rendement → anticipation → innovation

### Expérience 01 — Un même gain en DH, deux rendements différents

**Lien avec CH03 :** §§ 1–2.

**Pourquoi faisons-nous cela ?** Un changement absolu n’est comparable qu’une fois rapporté au prix de départ.

**Avant Python — sur le papier :** Un titre passe de 100 à 102 DH ; un autre de 1 000 à 1 002 DH. Calculez les deux pourcentages.

**À observer lorsque le calcul s'exécute :** Le gain de 2 DH est identique ; les rendements sont 2 % et 0,2 %. Le bon dénominateur est le prix initial.

In [ ]:
p0=np.array([100.,1000.]); p1=np.array([102.,1002.])
r=(p1/p0-1)*100
print(pd.DataFrame({'prix_initial_DH':p0,'prix_final_DH':p1,'rendement_pct':r.round(3)}))
assert np.allclose(r,[2,0.2])

**À vous de raisonner.** Que changerait une baisse de 2 DH ?

### Expérience 02 — Calculer une série de rendements et vérifier l’unité

**Lien avec CH03 :** § 2.

**Pourquoi faisons-nous cela ?** Nous passons de quelques prix aux observations journalières.

**Avant Python — sur le papier :** Avec 100 puis 101, puis 99, quel est le deuxième rendement ? Attention : le dénominateur est 101.

**À observer lorsque le calcul s'exécute :** Un prix est un **niveau**, le rendement est une **variation relative**. Les deux axes ne répondent pas à la même question.

In [ ]:
prix=np.array([100.,101.,99.,100.])
rpct=100*(prix[1:]/prix[:-1]-1)
print('Prix :',prix)
print('Rendements simples (%) :',np.round(rpct,4))
fig,ax=plt.subplots(1,2,figsize=(11,3));ax[0].plot(range(len(prix)),prix,'o-',color=COLOR)
ax[1].bar(range(1,len(prix)),rpct,color=[ACCENT if x<0 else COLOR for x in rpct])
ax[0].set(xlabel='Date (indice)',ylabel='Prix (DH)',title='Niveaux de prix')
ax[1].set(xlabel='Séance',ylabel='Rendement (%)',title='Variations relatives');plt.show()

### Expérience 03 — La prévision est-elle l’observation ?

**Lien avec CH03 :** § 3.

**Pourquoi faisons-nous cela ?** Une innovation est ce qui reste lorsqu’on retranche à la réalité ce qui était anticipé.

**Avant Python — sur le papier :** Si un rendement prévu est +0,2 % mais que le rendement observé est −1,3 %, calculez la surprise.

**À observer lorsque le calcul s'exécute :** La surprise vaut −1,5 point de pourcentage : un choc inattendu, différent de la variation observée elle-même.

In [ ]:
prevision=0.2; observe=-1.3
surprise=observe-prevision
print(f'Prévision : {prevision:+.2f} % | Observé : {observe:+.2f} % | Surprise : {surprise:+.2f} point de %')
assert np.isclose(surprise,-1.5)

> **Arrêt de compréhension**  
> Une surprise positive signifie-t-elle nécessairement un rendement positif ?

---
## II · Scénarios et moyenne conditionnelle

*Dans le cours : §§ 4–8*  

> **La question qui ouvre cette séquence**  
> Comment une même liste de rendements possibles peut-elle produire deux prévisions différentes ?

**Notre itinéraire :** scénarios → pondérations → information → notation → vérification

### Expérience 04 — La moyenne pondérée, avant toute formule abstraite

**Lien avec CH03 :** §§ 5–6.

**Pourquoi faisons-nous cela ?** Tous les scénarios ne sont pas forcément aussi plausibles.

**Avant Python — sur le papier :** Scénarios −2 %, 0 %, +3 % ; probabilités 20 %, 50 %, 30 %. Additionnez rendement × probabilité.

**À observer lorsque le calcul s'exécute :** L’espérance vaut +0,5 %. Ce n’est pas une promesse d’obtenir +0,5 % demain : aucun des scénarios ne vaut nécessairement cette moyenne.

In [ ]:
x=np.array([-2.,0.,3.]); proba=np.array([.2,.5,.3])
contributions=x*proba
print(pd.DataFrame({'rendement_pct':x,'probabilite':proba,'contribution':contributions}))
print('Somme des probabilités :',proba.sum(),'| Espérance :',contributions.sum(),'%')
assert np.isclose(proba.sum(),1)
assert np.isclose(contributions.sum(),.5)

### Expérience 05 — Quand l’information change les probabilités

**Lien avec CH03 :** §§ 4–7.

**Pourquoi faisons-nous cela ?** Les scénarios possibles peuvent rester identiques tandis que leur plausibilité change.

**Avant Python — sur le papier :** Réemployez −2 %, 0 %, +3 %. En cas d’annonce inquiétante, utilisez 50 %, 40 %, 10 %. Que devient la moyenne ?

**À observer lorsque le calcul s'exécute :** La moyenne passe de +0,5 % à −0,7 %. Ce n’est pas le passé qui change, mais l’information utilisée pour pondérer les scénarios.

In [ ]:
scenarios=np.array([-2.,0.,3.])
prob_norm=np.array([.2,.5,.3]);prob_alerte=np.array([.5,.4,.1])
table=pd.DataFrame({'rendement_pct':scenarios,'p_jour_ordinaire':prob_norm,
                    'p_annonce':prob_alerte})
display(table)
for nom,p in [('Jour ordinaire',prob_norm),('Jour d’annonce',prob_alerte)]:
 print(f'{nom} : espérance = {np.dot(scenarios,p):+.2f} %')
assert np.isclose(np.dot(scenarios,prob_alerte),-.7)

### Interprétation exacte des symboles — ils ne sont plus mystérieux

Nous avons maintenant effectué le calcul à la main. La notation peut donc arriver :

\[
\mu_t=\mathbb E(r_t\mid\mathcal F_{t-1}).
\]

- **\(r_t\)** : rendement de la séance \(t\), qui n'est pas encore connu au moment de prévoir ;
- **\(\mathcal F_{t-1}\)** : *l'ensemble des informations disponibles à la fin de la séance précédente* (cours, annonces déjà connues, calendrier, etc.) ;
- **\(\mid\)** : « sachant que » ou « compte tenu de » ;
- **\(\mathbb E\)** : moyenne des résultats possibles, pondérée par leur probabilité ;
- **\(\mu_t\)** : moyenne anticipée du rendement conditionnellement à ce dossier d'informations.

La filtration \(\mathcal F\) n'est pas une colonne magique : c'est une façon mathématique de déclarer **quelles données avaient le droit d'être connues au moment de prévoir**. Un cours de demain n'y figure pas.

### Expérience 06 — Pourquoi la surprise a une moyenne conditionnelle nulle — si la prévision est exacte

**Lien avec CH03 :** § 8.

**Pourquoi faisons-nous cela ?** L’identité \(\mathbb E[r_t-\mu_t\mid\mathcal F_{t-1}]=0\) vient de la définition de la moyenne conditionnelle ; elle ne garantit pas que nos modèles empiriques soient exacts.

**Avant Python — sur le papier :** Soustrayez la moyenne +0,5 % aux scénarios −2, 0, +3 et reprenez les pondérations ordinaires.

**À observer lorsque le calcul s'exécute :** La nullité vaut pour la vraie espérance conditionnelle, pas automatiquement pour une prévision estimée imparfaite.

In [ ]:
r=np.array([-2.,0.,3.]);p=np.array([.2,.5,.3]);mu=np.dot(r,p)
epsilon=r-mu
print(pd.DataFrame({'r_pct':r,'surprise_pct':epsilon,'probabilite':p}))
print('Moyenne pondérée des surprises :',np.dot(epsilon,p))
assert abs(np.dot(epsilon,p))<1e-12
mauvaise_prevision=1.0
print('Si nous prédisons incorrectement 1 %, la moyenne des erreurs =',np.dot(r-mauvaise_prevision,p),'%')

> **Arrêt de compréhension**  
> Peut-on calculer la vraie espérance conditionnelle d’un marché réel en énumérant tous ses scénarios ?

---
## III · Dispersion, variance et volatilité

*Dans le cours : §§ 9–19*  

> **La question qui ouvre cette séquence**  
> Deux actifs ont une moyenne zéro : peut-on les dire également risqués ?

**Notre itinéraire :** écarts → problème des signes → carrés → variance → écart-type → information

### Expérience 07 — Pourquoi la moyenne des écarts ne mesure rien

**Lien avec CH03 :** §§ 9–11.

**Pourquoi faisons-nous cela ?** Les écarts négatifs et positifs s’annulent quand la moyenne est correctement calculée.

**Avant Python — sur le papier :** Actif A : −1 %, +1 % ; actif B : −5 %, +5 %, équiprobables. Faites la moyenne des écarts.

**À observer lorsque le calcul s'exécute :** Les moyennes des écarts signés sont identiquement nulles ; les distances absolues distinguent déjà les deux actifs.

In [ ]:
A=np.array([-1.,1.]); B=np.array([-5.,5.]);
for nom,s in [('A',A),('B',B)]:
 print(nom,': moyenne =',s.mean(), '| moyenne des écarts =',np.mean(s-s.mean()),
       '| moyenne des distances absolues =',np.mean(abs(s-s.mean())))

### Expérience 08 — Fabriquer la variance sans boîte noire

**Lien avec CH03 :** §§ 12–13.

**Pourquoi faisons-nous cela ?** Élever les écarts au carré empêche les signes de se compenser et pénalise davantage les grandes amplitudes.

**Avant Python — sur le papier :** Pour A et B, écrivez tous les carrés et leurs moyennes. Prenez ensuite la racine carrée.

**À observer lorsque le calcul s'exécute :** La variance est en \(%^2\), la volatilité en \(%\). Une volatilité de 5 % **n’est pas un plafond des pertes**.

In [ ]:
table=[]
for nom,x in [('A',np.array([-1.,1.])),('B',np.array([-5.,5.]))]:
 mu=x.mean(); ecarts=x-mu; variance=np.mean(ecarts**2); volatilite=np.sqrt(variance)
 table.append([nom,mu,list(ecarts),list(ecarts**2),variance,volatilite])
display(pd.DataFrame(table,columns=['actif','moyenne_%','écarts_%','carrés_%²','variance_%²','volatilité_%']))
assert table[0][-1]==1 and table[1][-1]==5

### Expérience 09 — Trois scénarios inégaux : variance pondérée

**Lien avec CH03 :** §§ 14–16.

**Pourquoi faisons-nous cela ?** La variance conditionnelle utilise les mêmes probabilités que la moyenne conditionnelle.

**Avant Python — sur le papier :** Scénarios −2, 0, +3 %, probabilités 0,2 / 0,5 / 0,3. Moyenne 0,5 % : calculez les trois carrés des écarts.

**À observer lorsque le calcul s'exécute :** La formule \(\sum p_i(r_i-\mu)^2\) apparaît comme le résumé des contributions ligne par ligne.

In [ ]:
valeurs=np.array([-2.,0.,3.]); p=np.array([.2,.5,.3]);mu=np.dot(p,valeurs)
ec=valeurs-mu; contribution=p*ec**2
print(pd.DataFrame({'scenario_pct':valeurs,'ecart_pct':ec,'carre_pct2':ec**2,'proba':p,'contribution':contribution}))
print(f'Espérance {mu:.3f} %; variance {contribution.sum():.3f} %²; volatilité {np.sqrt(contribution.sum()):.3f} %')
assert np.isclose(contribution.sum(),3.25)

### Expérience 10 — Même moyenne, nouvelle information, nouveau risque

**Lien avec CH03 :** §§ 15–19.

**Pourquoi faisons-nous cela ?** Le conditionnement peut affecter la dispersion même si la moyenne reste inchangée.

**Avant Python — sur le papier :** Jour calme : −0,5 % ou +0,5 % ; jour agité : −3 % ou +3 %, chaque fois 50/50. Comparez variances.

**À observer lorsque le calcul s'exécute :** La moyenne vaut zéro dans les deux cas, mais la volatilité conditionnelle passe de 0,5 % à 3 %.

In [ ]:
dists={'Jour calme':np.array([-.5,.5]),'Jour agité':np.array([-3.,3.])}
for jour,x in dists.items():
 p=np.array([.5,.5]);mu=p@x;h=p@((x-mu)**2)
 print(f'{jour:<15} moyenne={mu:+.2f} %, variance={h:.2f} %², volatilité={np.sqrt(h):.2f} %')
fig,ax=plt.subplots(); ax.boxplot(list(dists.values()),tick_labels=list(dists.keys()),vert=True)
ax.set(ylabel='Rendement possible (%)',title='Même centre, amplitudes différentes (exemple construit)');plt.show()

**À vous de raisonner.** Une volatilité plus élevée signifie-t-elle que le rendement moyen baisse ?

### La formule peut maintenant apparaître

\[
\underbrace{\sigma_t^2}_{\substack{\text{variance prévue}\\\text{pour la séance }t}}
=\underbrace{\mathbb E\left[(r_t-\mu_t)^2\mid\mathcal F_{t-1}\right]}_{\text{moyenne des carrés des surprises, compte tenu des informations}}.
\]

L'**écart-type conditionnel** est \(\sigma_t=\sqrt{\sigma_t^2}\). En finance, on l'appelle souvent **volatilité conditionnelle**.

Ce n'est **pas** une mesure complète du risque : elle ne précise pas à elle seule les asymétries, les pertes extrêmes ou la liquidité. Elle apporte toutefois une composante essentielle de l'incertitude, que nous allons apprendre à prévoir.

**Attention aux unités :** rendements en % → variance en %² → volatilité en %. Le carré d'un rendement de 2 % vaut **4 en unités de %²**, mais **0,0004** si l'on code le rendement en décimal (`0.02`).

---
## IV · Quand l’amplitude garde la mémoire du passé

*Dans le cours : §§ 20–22, 30–31*  

> **La question qui ouvre cette séquence**  
> Si les signes changent presque au hasard, pourquoi les grosses fluctuations arrivent-elles souvent par groupes ?

**Notre itinéraire :** observer → comparer ACF → calculer une corrélation → tester ARCH → retirer une saisonnalité

### Expérience 11 — Voir les groupes de turbulences sur le marché fictif

**Lien avec CH03 :** § 20.

**Pourquoi faisons-nous cela ?** Une longue série révèle des séquences qui ne sont pas visibles sur trois ou quatre jours.

**Avant Python — sur le papier :** Prenez mentalement deux périodes : une calme, une agitée. Le signe des rendements indique-t-il directement l’amplitude ?

**À observer lorsque le calcul s'exécute :** Les signes sont difficiles à anticiper, mais les amplitudes fortes ont tendance à se regrouper. Le graphique seul ne suffit pas à établir un effet ARCH.

In [ ]:
t=marche.iloc[:650]
fig,axs=plt.subplots(2,1,figsize=(11,6),sharex=True)
axs[0].plot(t.date,t.rendement_pct,color=COLOR,lw=.8)
axs[1].plot(t.date,t.rendement_pct**2,color=ACCENT,lw=.8)
axs[0].set(ylabel='Rendement (%)',title='Signes des rendements : l’amplitude peut se regrouper')
axs[1].set(xlabel='Date',ylabel='Rendement² (%²)',title='Carrés : la mémoire des amplitudes devient visible')
plt.tight_layout();plt.show()

### Expérience 12 — Autocorrélation : la calculer avant de la tracer

**Lien avec CH03 :** § 21.

**Pourquoi faisons-nous cela ?** Une autocorrélation compare une série à elle-même décalée : pour un retard de 1, chaque valeur est comparée à celle de la veille.

**Avant Python — sur le papier :** Pour la suite 2, 4, 3, 5, 4, 6, écrivez les couples (valeur d’hier, valeur d’aujourd’hui).

**À observer lorsque le calcul s'exécute :** Le coefficient de corrélation entre deux sous-séries **n’est pas numériquement identique** à la convention usuelle d’ACF empirique. Nous affichons les deux pour éviter une confusion fréquente.

In [ ]:
x=np.array([2.,4.,3.,5.,4.,6.]);retard=1
avant=x[:-retard];apres=x[retard:]
print('Paires retardées :',list(zip(avant,apres)))
print('Corrélation des deux sous-séries (exemple didactique) :',np.corrcoef(avant,apres)[0,1])
# L'autocorrélation empirique usuelle utilise une convention différente :
def acf_empirique(x,k):
 x=np.asarray(x,dtype=float);c=x-x.mean()
 return float(np.dot(c[k:],c[:-k])/np.dot(c,c)) if k else 1.
print('ACF empirique conventionnelle au retard 1 :',acf_empirique(x,1))

### Expérience 13 — ACF des rendements contre ACF de leurs carrés

**Lien avec CH03 :** §§ 20–21.

**Pourquoi faisons-nous cela ?** Une série peut avoir une dépendance linéaire faible mais une dépendance nette des amplitudes.

**Avant Python — sur le papier :** Devinez quelle ACF devrait persister : celle du signe des rendements, ou celle de leurs carrés ?

**À observer lorsque le calcul s'exécute :** Les bandes ±1,96/√n ne sont qu’un **repère asymptotique approximatif sous bruit blanc**, pas un test universel ni une garantie pour les carrés conditionnellement hétéroscédastiques.

In [ ]:
rr=marche.rendement_pct.to_numpy()
lags=np.arange(1,26)
acf_r=np.array([acf_empirique(rr,k) for k in lags])
acf_sq=np.array([acf_empirique((rr-rr.mean())**2,k) for k in lags])
fig,axs=plt.subplots(1,2,figsize=(11,3.5),sharey=True)
for ax,y,title_ in [(axs[0],acf_r,'Rendements'),(axs[1],acf_sq,'Innovations au carré')]:
 ax.bar(lags,y,color=COLOR if title_=='Rendements' else ACCENT)
 ax.axhline(1.96/np.sqrt(len(rr)),ls='--',color=MUTED);ax.axhline(-1.96/np.sqrt(len(rr)),ls='--',color=MUTED)
 ax.set(xlabel='Retard (séances)',ylabel='ACF empirique',title=title_)
plt.tight_layout();plt.show()
print('Bandes ±1,96/√n =',round(1.96/np.sqrt(len(rr)),4))

### Expérience 14 — Distinguer une saisonnalité déterministe de la dépendance ARCH

**Lien avec CH03 :** §§ 20–22.

**Pourquoi faisons-nous cela ?** Une volatilité qui augmente mécaniquement à l’ouverture et à la clôture peut produire une structure régulière qui n’est pas nécessairement un effet GARCH.

**Avant Python — sur le papier :** Imaginez cinq créneaux intrajournaliers, dont le premier et le dernier sont systématiquement plus agités. Comment les comparer équitablement ?

**À observer lorsque le calcul s'exécute :** La forme en U provient ici d’un **profil déterministe imposé**. Il serait abusif de qualifier tout motif de ce type de persistance stochastique GARCH.

In [ ]:
rng=np.random.default_rng(SEED+9);njours=80
creneaux=np.tile(np.arange(5),njours)
profil=np.array([1.8,1.1,.75,1.0,1.7])
retour=rng.normal(0,1,len(creneaux))*profil[creneaux]
df_intra=pd.DataFrame({'créneau':creneaux+1,'r²':retour**2,'r':retour})
moy_carres=df_intra.groupby('créneau')['r²'].mean()
fig,ax=plt.subplots();ax.bar(moy_carres.index,moy_carres.values,color=COLOR)
ax.set(xlabel='Créneau dans la séance (1 = ouverture, 5 = clôture)',ylabel='Moyenne de r² (%²)',title='Saisonnalité intrajournalière artificielle');plt.show()
normalise=retour/profil[creneaux]
print('Variances par créneau, après retrait du profil connu :',df_intra.assign(z=normalise).groupby('créneau').z.var().round(2).to_dict())

### Expérience 15 — Test ARCH-LM : une régression sur les carrés

**Lien avec CH03 :** § 31.

**Pourquoi faisons-nous cela ?** On demande si les carrés des résidus sont expliqués par leurs propres retards, sous des hypothèses asymptotiques.

**Avant Python — sur le papier :** Hypothèse nulle : aucune dépendance ARCH aux 5 retards retenus. Que signifierait une très petite p-value ?

**À observer lorsque le calcul s'exécute :** Une petite p-value suggère un rejet de l’absence d’effet ARCH d’ordre 5. Elle ne mesure pas « la probabilité que H0 soit vraie » et ne prouve pas qu’un GARCH précis soit le bon modèle.

In [ ]:
r0=rr-rr.mean();q=5
# Fonction de diagnostic reconnue : retourne statistique LM et p-value
stat_lm,pvalue_lm,stat_f,pvalue_f=het_arch(r0,nlags=q)
print(f'ARCH-LM({q}) : statistique={stat_lm:.2f}, p-value={pvalue_lm:.3g}')
print('Lecture : sous H0 et les hypothèses du test, une statistique au moins aussi extrême serait-elle fréquente ?')

> **Arrêt de compréhension**  
> Pourquoi calculer l’ARCH-LM sur les innovations d’un modèle de moyenne plutôt que sur des prix bruts ?

---
## V · ARCH puis GARCH : construire la mémoire de la variance

*Dans le cours : §§ 22–27*  

> **La question qui ouvre cette séquence**  
> Quel calcul relie l’ampleur du choc d’hier à la volatilité que nous anticipons demain ?

**Notre itinéraire :** récursion ARCH → insuffisance → GARCH → persistance → multi-horizon → régimes

### Expérience 16 — ARCH(1) : calculer trois scénarios à la main

**Lien avec CH03 :** § 22.

**Pourquoi faisons-nous cela ?** L’idée la plus directe est que la variance prévue demain augmente avec le carré du choc observé aujourd’hui.

**Avant Python — sur le papier :** Avec ω=0,20 et α=0,50 : calculez la variance prévue après un choc de 0, de +1, puis de −2.

**À observer lorsque le calcul s'exécute :** Le signe disparaît au carré : à amplitude égale, un ARCH symétrique répond de la même manière aux chocs positifs et négatifs.

In [ ]:
omega=.20;alpha=.50
for e in [0.,1.,-2.]:
 h=omega+alpha*e**2
 print(f'choc {e:+.1f} % -> variance prévue {h:.2f} %² ; volatilité {np.sqrt(h):.3f} %')
assert .20+alpha*(-2)**2==2.2

### Expérience 17 — GARCH(1,1) : deux mémoires au lieu d’une

**Lien avec CH03 :** §§ 23–24.

**Pourquoi faisons-nous cela ?** ARCH ne regarde que les anciens chocs ; GARCH ajoute la variance déjà prévue au tour précédent.

**Avant Python — sur le papier :** Avec ω=0,04, α=0,08, β=0,88, ε=−1,5 et h=1,20, calculez la nouvelle variance.

**À observer lorsque le calcul s'exécute :** Chaque terme a un sens : ω (socle), α (nouvelle information), β (héritage du risque passé). Ce sont des contributions à une **variance**, pas directement à une volatilité.

In [ ]:
om,al,be=.04,.08,.88
choct=-1.5;h_auj=1.20
termes={'niveau ω':om,'nouveau choc α ε²':al*choct**2,'mémoire β h':be*h_auj}
print(pd.Series(termes).round(4));h_dem=sum(termes.values())
print(f'Somme = {h_dem:.3f} %² ; volatilité = {np.sqrt(h_dem):.3f} %')
fig,ax=plt.subplots(figsize=(9,2.8));ax.barh(list(termes),list(termes.values()),color=[MUTED,ACCENT,COLOR]);ax.set(xlabel='Contribution à la variance (%²)',title='Décomposition du prochain risque');plt.show()
assert np.isclose(h_dem,1.276)

### Expérience 18 — Faire varier la mémoire sans dépendre de widgets

**Lien avec CH03 :** §§ 24–26.

**Pourquoi faisons-nous cela ?** La persistance \(p=\alpha+\beta\) gouverne la décroissance de l’écart prévu à la variance de long terme dans un GARCH standard à variance finie.

**Avant Python — sur le papier :** Comparez p=0,70, 0,90 et 0,98 : quelle courbe doit revenir le plus lentement vers le niveau de long terme ?

**À observer lorsque le calcul s'exécute :** Les courbes représentent une différence de variance qui suit \(p^k\). La demi-vie est celle de **l’écart de variance**, pas exactement celle de la volatilité après racine carrée.

In [ ]:
k=np.arange(0,81); depart=3.;long_terme=1.
fig,ax=plt.subplots()
for p in [.70,.90,.98]:
 h=long_terme+p**k*(depart-long_terme)
 ax.plot(k,np.sqrt(h),label=f'persistance p={p:.2f}')
ax.axhline(1,color=MUTED,ls='--',label='volatilité LT = 1 %')
ax.set(xlabel='Séances après le choc',ylabel='Volatilité prévue (%)',title='Même choc, trois vitesses d’oubli');ax.legend();plt.show()
for p in [.70,.90,.98]:print(f'p={p}: demi-vie des écarts de variance ≈ {np.log(.5)/np.log(p):.2f} séances')

### Démonstration de la variance de long terme

Partons de la formule GARCH(1,1) :

\[
h_t=\omega+\alpha\varepsilon_{t-1}^2+\beta h_{t-1}.
\]

Dans un régime stationnaire **avec variance finie** et une innovation standardisée de variance 1, nous avons \(\mathbb E[\varepsilon_{t-1}^2]=\mathbb E[h_{t-1}]=\bar h\). En prenant l'espérance des deux côtés :

\[
\bar h=\omega+(\alpha+\beta)\bar h
\quad\Longrightarrow\quad
\bar h=\frac{\omega}{1-\alpha-\beta},\qquad \alpha+\beta<1.
\]

La contrainte \(\alpha+\beta<1\) assure ici la finitude de cette variance de long terme (avec les autres hypothèses du modèle). À \(\alpha+\beta=1\), nous rencontrons la frontière IGARCH : **on ne doit pas calculer \(\omega/(1-p)\)** en mettant arbitrairement une petite valeur au dénominateur.

### Expérience 19 — Prévision GARCH à 1, 2 et 10 séances

**Lien avec CH03 :** § 27.

**Pourquoi faisons-nous cela ?** Au-delà du premier jour, le choc futur n’est pas connu : on remplace son carré par sa valeur conditionnelle moyenne, la variance prévue.

**Avant Python — sur le papier :** Supposez h demain=1,276, ω=0,04 et p=0,96. Calculez la variance du surlendemain.

**À observer lorsque le calcul s'exécute :** La formule repose sur un GARCH stationnaire symétrique. À plusieurs jours, on **prévoit les carrés des innovations futures**, on ne les connaît pas.

In [ ]:
omega=.04;p=.96;h1=1.276
n=20;horizon=np.arange(1,n+1)
long_term=omega/(1-p)
previsions=long_term+p**(horizon-1)*(h1-long_term)
print(pd.DataFrame({'horizon':horizon[:5],'variance_pct2':previsions[:5],'volatilite_pct':np.sqrt(previsions[:5])}).round(4))
fig,ax=plt.subplots();ax.plot(horizon,previsions,'o-',color=COLOR,label='Variance prévue')
ax.axhline(long_term,ls='--',color=ACCENT,label='Variance de long terme');ax.set(xlabel='Horizon (séances)',ylabel='Variance prévue (%²)',title='Retour progressif vers le niveau de long terme');ax.legend();plt.show()
assert np.isclose(previsions[1],.04+.96*1.276)

**À vous de raisonner.** Peut-on additionner directement des volatilités quotidiennes ?

### Expérience 20 — Rupture de régime : ne pas confondre persistance et changement d’époque

**Lien avec CH03 :** § 26 et § 45.

**Pourquoi faisons-nous cela ?** Un modèle stationnaire peut attribuer une persistance excessive à un saut permanent de niveau.

**Avant Python — sur le papier :** Produisez 500 rendements indépendants avec σ=0,5 % puis 500 avec σ=2,0 %. Est-ce une mémoire des chocs ou une rupture imposée ?

**À observer lorsque le calcul s'exécute :** La mémoire visuelle des carrés peut résulter d’une rupture non modélisée. Un résultat ARCH-LM ou une forte persistance GARCH ne permet pas, seul, d’identifier la vraie cause économique.

In [ ]:
rng=np.random.default_rng(SEED+20)
serie=np.r_[rng.normal(0,.5,500),rng.normal(0,2.,500)]
fig,ax=plt.subplots();ax.plot(serie,color=COLOR,lw=.75);ax.axvline(500,color=ACCENT,ls='--')
ax.set(xlabel='Séance simulée',ylabel='Rendement (%)',title='Deux régimes imposés, sans récursion GARCH');plt.show()
print('Volatilité première moitié :',np.std(serie[:500]).round(3),'%; deuxième moitié :',np.std(serie[500:]).round(3),'%')

---
## VI · Estimer plutôt que choisir les paramètres arbitrairement

*Dans le cours : §§ 28–29 et § 40*  

> **La question qui ouvre cette séquence**  
> Si les paramètres ω, α et β nous sont inconnus, comment les apprendre sans consulter les données futures ?

**Notre itinéraire :** densité → vraisemblance → logarithme → optimisation → contrôle → séparation temporelle

### Le sens de la vraisemblance, sans vocabulaire opaque

Imaginons que nous ayons observé trois erreurs de prévision proches de zéro. Deux modèles proposent des variances différentes. Un modèle très agité attribue moins de **densité** aux observations proches de zéro qu'un modèle plus concentré. On mesure cela par une **densité de probabilité**, pas par la probabilité d'un point isolé (qui vaut zéro dans un modèle continu).

Si \(\varepsilon_t\) suit une loi normale conditionnelle centrée de variance \(h_t>0\), sa densité observée est

\[
 f(\varepsilon_t\mid h_t)=\frac{1}{\sqrt{2\pi h_t}}\exp\left(-\frac{\varepsilon_t^2}{2h_t}\right).
\]

**D'où vient le logarithme ?** Pour plusieurs dates, on multiplie leurs densités **conditionnelles**. Prendre le logarithme transforme le produit en somme, plus facile à calculer numériquement :

\[
-\log L=\frac12\sum_t\left[\log(2\pi)+\log h_t+\frac{\varepsilon_t^2}{h_t}\right].
\]

La formule n'a pas été « devinée » : c'est simplement le logarithme négatif d'une densité normale, additionné au fil du temps.

### Expérience 21 — Trois observations, deux niveaux de variance : pourquoi l’optimiseur a-t-il un travail ?

**Lien avec CH03 :** § 29.

**Pourquoi faisons-nous cela ?** Nous découvrons une fonction objectif en comparant des modèles élémentaires à variance constante.

**Avant Python — sur le papier :** Pour les surprises 0,1 ; −0,2 ; 0,05 %, comparer h=0,04 et h=4 en %².

**À observer lorsque le calcul s'exécute :** Les scores sont calculés sur des densités ; une NLL peut être négative pour des unités continues, et ce n’est pas une erreur.

In [ ]:
observ=np.array([.1,-.2,.05])
def nll_normale(e,h):
 e=np.asarray(e);h=np.asarray(h)
 return .5*np.sum(np.log(2*np.pi*h)+e**2/h)
for h in [.04,4.]:
 detail=.5*(np.log(2*np.pi*h)+observ**2/h)
 print('h =',h,'| contributions =',np.round(detail,3),'| total NLL =',round(detail.sum(),3))
print('NLL = moins log-vraisemblance ; plus petite est meilleure sur ces observations.')

### Se protéger avant d'estimer : les frontières temporelles

Notre marché fictif contient 2 400 séances. Nous décidons **avant toute évaluation** :

- **Train : 60 %**, pour apprendre les paramètres ;
- **Validation : 20 %**, pour comparer des variantes ;
- **Test : 20 %**, **intouchable** jusqu'à la comparaison finale.

Nous ne mélangerons jamais les dates. Une moyenne, un écart-type ou un paramètre estimé sur les observations futures créerait une fuite de données. En revanche, lors d'une prévision *one-step*, une fois que la séance \(t-1\) est observée, son rendement peut légalement être utilisé pour prévoir \(t\).

In [ ]:
n=len(marche);fin_train=int(.6*n);fin_validation=int(.8*n)
train=marche.iloc[:fin_train].copy(); validation=marche.iloc[fin_train:fin_validation].copy();test=marche.iloc[fin_validation:].copy()
for nom,bloc in [('TRAIN',train),('VALIDATION',validation),('TEST',test)]:
 print(f'{nom:<11}{len(bloc):4d} séances | {bloc.date.iloc[0].date()} → {bloc.date.iloc[-1].date()}')
assert train.date.max()<validation.date.min()<test.date.min()
mu_train=float(train.rendement_pct.mean())
eps_train=train.rendement_pct.to_numpy()-mu_train
print('Moyenne estimée exclusivement sur train :',round(mu_train,4),'%')

### Notre propre filtre GARCH : ce que fait réellement la récursion

Pour que le logiciel ne soit pas une boîte noire, nous codons une fonction qui parcourt les observations dans l'ordre. La valeur \(h_t\) est calculée à partir de \(\varepsilon_{t-1}\) et \(h_{t-1}\) **avant** de regarder \(\varepsilon_t\). Un éventuel terme \(\gamma 1_{\{\varepsilon_{t-1}<0\}}\varepsilon_{t-1}^2\) permet ensuite une extension asymétrique. Lorsque \(\gamma=0\), nous retrouvons GARCH.

L'initialisation utilise une variance du bloc d'entraînement. Les premières observations peuvent être influencées par ce choix ; ici nous utilisons un « échauffement » long, et l'analyse hors échantillon ne dépend pas des quelques premiers pas.

In [ ]:
def filtrer_variance(eps, omega, alpha, beta, gamma=0., initial=None):
    eps=np.asarray(eps,dtype=float)
    assert omega>0 and alpha>=0 and beta>=0 and gamma>=0
    h=np.empty(len(eps),dtype=float)
    h[0]=float(np.var(eps[:min(300,len(eps))]) if initial is None else initial)
    h[0]=max(h[0],1e-8)
    for t in range(1,len(eps)):
        negatif=1. if eps[t-1]<0 else 0.
        h[t]=omega+(alpha+gamma*negatif)*eps[t-1]**2+beta*h[t-1]
    return h

def decoder(theta, asymmetric=False):
    # Les transformations garantissent omega>0 et une persistance < 0,995.
    omega=np.exp(theta[0])
    alpha=.30*expit(theta[1])
    if asymmetric:
        gamma=.35*expit(theta[2])
        beta=(.995-alpha-.5*gamma)*expit(theta[3])
    else:
        gamma=0.
        beta=(.995-alpha)*expit(theta[2])
    return float(omega),float(alpha),float(beta),float(gamma)

def estimer_garch(eps,asymmetric=False,maxiter=320):
    eps=np.asarray(eps,dtype=float)
    x0=np.array([np.log(.04),-1.1, -1.2, 2.0] if asymmetric else [np.log(.04),-1.1, 2.0])
    def fonction(theta):
        pars=decoder(theta,asymmetric)
        h=filtrer_variance(eps,*pars[:3],gamma=pars[3])
        return nll_normale(eps,h)
    sol=minimize(fonction,x0,method='L-BFGS-B',options={'maxiter':maxiter,'ftol':1e-9})
    params=decoder(sol.x,asymmetric)
    return {'params':params,'objective':float(sol.fun),'success':bool(sol.success),
            'message':str(sol.message),'iterations':int(sol.nit),'model':'GJR' if asymmetric else 'GARCH'}

fit_garch=estimer_garch(eps_train,asymmetric=False)
print('Estimation normale GARCH sur TRAIN :')
print(pd.Series(dict(zip(['omega','alpha','beta','gamma'],fit_garch['params']))).round(5))
print('Convergence :',fit_garch['success'],'| iterations :',fit_garch['iterations'],'| message :',fit_garch['message'])
assert fit_garch['success'], 'L’optimisation GARCH doit converger avant interprétation.'
assert fit_garch['params'][1]+fit_garch['params'][2]<1

### Expérience 22 — QMLE : comment contrôler ce que l’optimiseur nous rend

**Lien avec CH03 :** §§ 28–30.

**Pourquoi faisons-nous cela ?** La vraisemblance normale peut estimer la dynamique de variance même si la loi vraie n’est pas normale, sous des conditions théoriques précises. Ici nous vérifions, au moins, contraintes, convergence et cohérence.

**Avant Python — sur le papier :** Comparez α+β à 1. Une persistance proche de 1 signifie-t-elle forcément une mémoire infinie ?

**À observer lorsque le calcul s'exécute :** Le modèle est **mal spécifié** par rapport au générateur : symétrique plutôt qu’asymétrique, loi normale plutôt que Student-t. Une convergence numérique ne prouve jamais l’adéquation scientifique.

In [ ]:
om,al,be,ga=fit_garch['params'];p=al+be
h_fit_train=filtrer_variance(eps_train,om,al,be)
print('Paramètres :',{'omega':round(om,5),'alpha':round(al,5),'beta':round(be,5)})
print('Persistance :',round(p,5),' | variance LT :',round(om/(1-p),4),'%²')
print('Vraie dynamique (GJR, asymétrique) :',VRAIS_PARAMETRES)
fig,ax=plt.subplots(figsize=(11,3.8))
ax.plot(train.date.iloc[-350:],np.sqrt(h_fit_train[-350:]),label='GARCH estimé (σ)',color=COLOR)
ax.plot(train.date.iloc[-350:],train.volatilite_latente_pct.iloc[-350:],label='Volatilité vraie simulée',color=ACCENT,alpha=.8)
ax.set(xlabel='Date TRAIN',ylabel='Volatilité (%)',title='La variance latente sert d’oracle pédagogique, pas de variable d’entrée');ax.legend();plt.show()

> **Arrêt de compréhension**  
> Peut-on comparer deux log-vraisemblances calculées sur des jeux de données différents et conclure automatiquement ?

---
## VII · Tester les limites : résidus, queues, asymétries

*Dans le cours : §§ 30–34 et § 45*  

> **La question qui ouvre cette séquence**  
> Le logiciel a trouvé des paramètres. Comment savoir si le modèle a vraiment compris les fluctuations ?

**Notre itinéraire :** résidus standardisés → diagnostics → Student-t → choc signé → GJR → EGARCH

### Expérience 23 — Résidus standardisés : ce que GARCH aurait dû neutraliser

**Lien avec CH03 :** § 30.

**Pourquoi faisons-nous cela ?** Une fois divisé par la volatilité prévue, un choc ne devrait plus présenter une forte mémoire de variance si le modèle est correctement spécifié.

**Avant Python — sur le papier :** Si ε=−2 % et σ=1 %, combien vaut z=ε/σ ?

**À observer lorsque le calcul s'exécute :** L’amélioration attendue doit être examinée ; une p-value non significative ne prouve pas que le modèle soit vrai. Nous devons aussi examiner les queues de distribution et l’asymétrie.

In [ ]:
z_fit_train=eps_train/np.sqrt(h_fit_train)
print('Moyenne des z :',round(z_fit_train.mean(),4),'| variance des z :',round(z_fit_train.var(),4))
_,p_brut,*_=het_arch(eps_train,nlags=5)
_,p_res,*_=het_arch(z_fit_train,nlags=5)
print('ARCH-LM avant GARCH :',round(p_brut,8),'| après standardisation :',round(p_res,8))
fig,axs=plt.subplots(1,2,figsize=(11,3.5))
axs[0].plot(train.date.iloc[-400:],eps_train[-400:],color=ACCENT,lw=.8)
axs[1].plot(train.date.iloc[-400:],z_fit_train[-400:],color=COLOR,lw=.8)
axs[0].set(title='Innovations brutes',xlabel='Date',ylabel='ε (%)');axs[1].set(title='Innovations standardisées',xlabel='Date',ylabel='z (sans unité)')
plt.tight_layout();plt.show()

### Expérience 24 — Normale contre Student-t : les événements extrêmes

**Lien avec CH03 :** § 32.

**Pourquoi faisons-nous cela ?** Deux lois peuvent avoir moyenne 0 et variance 1, mais donner des probabilités différentes aux grands écarts.

**Avant Python — sur le papier :** Comparez les quantiles à 1 % d’une normale standard et d’une Student-t standardisée à ν=7.

**À observer lorsque le calcul s'exécute :** « Student-t standardisée » signifie variance unitaire : la Student-t brute a variance ν/(ν−2), il faut donc la multiplier par √((ν−2)/ν). La figure ne dit rien d’une probabilité de gain garantie.

In [ ]:
nu=7.;scale=np.sqrt((nu-2)/nu)
q_n=stats.norm.ppf(.01);q_t=scale*stats.t.ppf(.01,df=nu)
print(f'Quantile normal 1% : {q_n:.4f} | Student-t standardisée (ν=7) : {q_t:.4f}')
x=np.linspace(-5,5,501)
fig,ax=plt.subplots();ax.plot(x,stats.norm.pdf(x),label='Normale N(0,1)',color=COLOR)
ax.plot(x,stats.t.pdf(x/scale,nu)/scale,label='Student-t standardisée ν=7',color=ACCENT)
ax.set(xlabel='Innovation standardisée z',ylabel='Densité',title='Même variance unitaire, queues différentes');ax.legend();plt.show()

### Expérience 25 — Asymétrie : une baisse et une hausse de même amplitude

**Lien avec CH03 :** §§ 33–34.

**Pourquoi faisons-nous cela ?** Un GARCH symétrique attribue le même effet à +2 et −2. Un GJR autorise une réaction différente aux chocs négatifs.

**Avant Python — sur le papier :** GJR avec ω=.04, α=.07, γ=.10, β=.80 et h=1. Après +2 puis −2, quelle différence ?

**À observer lorsque le calcul s'exécute :** L’effet supplémentaire d’un choc négatif est γ ε². Il s’agit d’une asymétrie du **modèle de variance**, non d’une preuve de causalité économique.

In [ ]:
om=.04;al=.07;gamma=.10;beta=.80;h0=1.
for e in [2.,-2.]:
 h1=om+(al+gamma*(e<0))*e**2+beta*h0
 print(f'Choc {e:+.0f} % → h suivant = {h1:.3f} %² et σ = {np.sqrt(h1):.3f} %')
print('Différence de variance :',gamma*2**2,'%²')
assert np.isclose(gamma*2**2,.4)

### Expérience 26 — Estimer GJR et comparer sur validation, non sur test

**Lien avec CH03 :** § 33 et § 40.

**Pourquoi faisons-nous cela ?** Le modèle asymétrique ajoute un degré de liberté. Nous devons évaluer si cette complexité vaut son coût.

**Avant Python — sur le papier :** Avant d’exécuter : pourquoi une NLL d’entraînement plus faible ne suffit-elle pas à choisir définitivement GJR ?

**À observer lorsque le calcul s'exécute :** La sélection exploite uniquement train et validation. Les valeurs de validation ont été utilisées pour **choisir**, pas pour calculer initialement les paramètres du modèle. Le test reste réservé à la fin.

In [ ]:
fit_gjr=estimer_garch(eps_train,asymmetric=True,maxiter=400)
print('Convergence GJR :',fit_gjr['success'],'|',fit_gjr['message'])
print(pd.DataFrame([dict(model=m['model'],omega=m['params'][0],alpha=m['params'][1],
                          beta=m['params'][2],gamma=m['params'][3],nll_train=m['objective'])
                    for m in [fit_garch,fit_gjr]]).round(5))
assert fit_gjr['success'], 'La convergence de GJR est requise pour interpréter les scores.'
# Le passé antérieur reste utilisable pour établir l'état filtré.
r_passe=marche.rendement_pct.to_numpy()[:fin_validation]
e_passe=r_passe-mu_train
h_g=filtrer_variance(e_passe,*fit_garch['params'][:3],gamma=0,initial=np.var(eps_train))
h_j=filtrer_variance(e_passe,*fit_gjr['params'][:3],gamma=fit_gjr['params'][3],initial=np.var(eps_train))
sl_val=slice(fin_train,fin_validation)
score_g=nll_normale(e_passe[sl_val],h_g[sl_val])/len(e_passe[sl_val])
score_j=nll_normale(e_passe[sl_val],h_j[sl_val])/len(e_passe[sl_val])
print(f'NLL moyenne de validation : GARCH {score_g:.4f} | GJR {score_j:.4f}')
modele_choisi='GJR' if score_j<score_g else 'GARCH'
print('Choix strictement sur validation :',modele_choisi)

### Expérience 27 — EGARCH : comprendre l’intérêt du logarithme de variance

**Lien avec CH03 :** § 34.

**Pourquoi faisons-nous cela ?** Une autre idée consiste à modéliser log(h) pour garantir h>0 après exponentiation.

**Avant Python — sur le papier :** Évaluez un pas \(\log h_{t+1}=c+\beta\log h_t+\alpha(|z_t|-E|Z|)+\gamma z_t\), avec Z normal standard.

**À observer lorsque le calcul s'exécute :** Ici nous simulons **un seul pas EGARCH** pour comprendre la construction. Nous ne prétendons pas avoir effectué une estimation complète d’EGARCH.

In [ ]:
c=-.05;be=.92;al=.16;ga=-.08;h_actuelle=1.0
Ezabs=np.sqrt(2/np.pi)
for z in [2.,-2.]:
 logh=c+be*np.log(h_actuelle)+al*(abs(z)-Ezabs)+ga*z
 print(f'z={z:+.0f} : log variance prévue={logh:.4f}, variance={np.exp(logh):.4f} > 0')

### Discipline de modélisation

Une amélioration numérique sur un jeu de données construit à partir d'un GJR ne prouverait rien sur un marché réel. Le modèle choisi est celui qui minimise notre score **sur la validation**, sous le protocole défini à l'avance. Pour les risques extrêmes, une vraisemblance normale peut donner des prévisions de variance utiles tout en sous-estimant les queues : **variance conditionnelle et loi des innovations sont deux objets différents**.

---
## VIII · Du risque de fluctuation au risque de perte

*Dans le cours : §§ 35–39 et § 27*  

> **La question qui ouvre cette séquence**  
> Comment traduire une volatilité de 1,5 % en une somme de dirhams que la direction comprend ?

**Notre itinéraire :** convention des pertes → quantiles → VaR → ES → distribution → horizons

### Trois notions à ne jamais confondre

Supposons que l'on détienne **100 000 DH** d'un actif et que demain le rendement soit de \(-2\%\). La perte signée en dirhams vaut \(L=-V r/100=2\,000\,\mathrm{DH}\) (si l'on néglige frais et autres mécanismes). Le risque pertinent dans cette expérience est une **perte positive**. Nous adopterons partout cette convention :

\[
\text{Rendement positif }r>0\ \Rightarrow\ \text{perte signée }L<0,
\quad \text{et inversement.}
\]

La **VaR à 99 %** est un seuil tel que, sous notre modèle prédictif, **1 % de la masse des pertes se situe au-delà de ce seuil** (dans une loi continue et sous définition quantile appropriée). Ce n'est **pas** la perte maximale. L'**Expected Shortfall à 99 %** est la perte moyenne dans le pire 1 % de la distribution, lorsque la définition et l'intégrabilité le permettent. Nous allons construire les deux à partir de nombres concrets.

### Expérience 28 — Des rendements à la perte en dirhams

**Lien avec CH03 :** § 35–36.

**Pourquoi faisons-nous cela ?** Le signe change, et l’unité financière apparaît : ce n’est plus seulement un pourcentage abstrait.

**Avant Python — sur le papier :** Avec 100 000 DH, calculez le gain/perte pour +1 %, −2 % et −4 %.

**À observer lorsque le calcul s'exécute :** Cette conversion est valide pour une exposition unitaire simple. Elle ne couvre pas la liquidité, les coûts, la non-linéarité des dérivés ni le risque de défaut.

In [ ]:
valeur=100_000.
r_simples=np.array([1.,-2.,-4.])
pertes=-valeur*r_simples/100
print(pd.DataFrame({'rendement_pct':r_simples,'perte_signee_DH':pertes.astype(int)}))
print('Perte positive = argent perdu ; valeur négative = gain, sous notre convention.')

### Expérience 29 — VaR normale 99 % : dériver le seuil

**Lien avec CH03 :** § 36.

**Pourquoi faisons-nous cela ?** Si \(r=\mu+\sigma Z\), \(Z\sim\mathcal N(0,1)\), la partie basse 1 % des rendements correspond au quantile \(q_{0,01}\approx-2,326\).

**Avant Python — sur le papier :** Avec μ=0 %, σ=1,5 % et 100 000 DH, calculez le rendement seuil puis sa perte associée.

**À observer lorsque le calcul s'exécute :** La VaR calculée est positive, dans notre convention de perte. Dire « VaR 99 % ≈ 3 490 DH » ne signifie pas que les pertes au-delà sont impossibles.

In [ ]:
V=100_000.;mu=0.;sigma=1.5;p=.01
q=stats.norm.ppf(p);rendement_seuil=mu+sigma*q
var_DH=-V*rendement_seuil/100
print(f'Quantile normal à 1 % = {q:.4f}')
print(f'Seuil de rendement = {rendement_seuil:.4f} %')
print(f'VaR normale 99 % = {var_DH:,.2f} DH')
assert 3400<var_DH<3600

### D'où vient l'Expected Shortfall, sans formule magique ?

Prenons le même modèle \(r=\mu+\sigma Z\) et le quantile bas \(q_p\) tel que \(\mathbb P(Z\le q_p)=p\). Nous voulons la **moyenne des rendements dans les cas les plus défavorables**. Pour une normale standard, la densité est \(\varphi(z)=(2\pi)^{-1/2}e^{-z^2/2}\). Sa dérivée est \(\varphi'(z)=-z\varphi(z)\). Donc

\[
\int_{-\infty}^{q_p}z\varphi(z)\,dz=-\varphi(q_p)
\quad\Rightarrow\quad
\mathbb E[Z\mid Z\le q_p]=-\frac{\varphi(q_p)}{p}.
\]

La perte est \(L=-Vr/100\). Nous en déduisons **pour cette loi normale continue** :

\[
\boxed{ES_{1-p}=\frac{V}{100}\left(-\mu+\sigma\frac{\varphi(q_p)}{p}\right).}
\]

La formule explicite de l'ES dépend de la loi supposée. Nous reviendrons aux queues épaisses avec Student-t.

### Expérience 30 — VaR versus ES sur le même exemple

**Lien avec CH03 :** §§ 37–39.

**Pourquoi faisons-nous cela ?** VaR localise le seuil ; ES examine ce qui se passe au-delà.

**Avant Python — sur le papier :** Reprenez V=100 000 DH, σ=1,5 % et p=1 %. Calculez la densité normale en q puis l’ES.

**À observer lorsque le calcul s'exécute :** La VaR ne décrit pas la gravité des pertes à l’intérieur de la queue. L’ES en donne la moyenne, toujours **sous le modèle adopté**.

In [ ]:
densite=stats.norm.pdf(q)
es_DH=V/100*(-mu+sigma*densite/p)
print(f'Quantile q={q:.4f}; densité φ(q)={densite:.5f}')
print(f'VaR 99 % : {var_DH:,.2f} DH | ES 99 % : {es_DH:,.2f} DH')
fig,ax=plt.subplots(figsize=(10,3.7));zz=np.linspace(-5,5,600)
ax.plot(zz,stats.norm.pdf(zz),color=COLOR)
ax.fill_between(zz,stats.norm.pdf(zz),where=zz<=q,color=ACCENT,alpha=.55)
ax.axvline(q,color=ACCENT,ls='--',label=f'Seuil 1 % : {q:.2f}')
ax.set(xlabel='Rendement standardisé Z',ylabel='Densité',title='La VaR marque le seuil ; l’ES résume la zone colorée');ax.legend();plt.show()
assert es_DH>var_DH

### Expérience 31 — Student-t : normaliser avant de calculer le risque

**Lien avec CH03 :** §§ 32, 36–39.

**Pourquoi faisons-nous cela ?** Une Student-t à ν>2 possède une variance ν/(ν−2). Nos innovations conditionnelles exigent variance 1.

**Avant Python — sur le papier :** Pour ν=7, calculez le facteur \(s=\sqrt{5/7}\), puis le quantile bas \(s\,t_{ν}^{-1}(0,01)\).

**À observer lorsque le calcul s'exécute :** La loi choisie peut modifier les pertes extrêmes même à **volatilité identique**. Son estimation doit être faite sur le passé uniquement.

In [ ]:
nu=7.;s=np.sqrt((nu-2)/nu);qt=stats.t.ppf(.01,nu);qstandard=s*qt
# ES standardisée : intégrale analytique d'une Student-t gauche
E_z_bas=-s*(nu+qt**2)/(nu-1)*stats.t.pdf(qt,nu)/.01
VaR_t=-V/100*(mu+sigma*qstandard)
ES_t=-V/100*(mu+sigma*E_z_bas)
print(f'Student-t ν={nu:.0f}, facteur={s:.5f}, q brut={qt:.4f}, q standardisé={qstandard:.4f}')
print(f'VaR-t 99 % = {VaR_t:,.2f} DH | ES-t 99 % = {ES_t:,.2f} DH')
assert ES_t>VaR_t and VaR_t>0

### Expérience 32 — L’erreur du « × racine du temps » appliqué sans hypothèses

**Lien avec CH03 :** §§ 27, 35–39.

**Pourquoi faisons-nous cela ?** Pour des innovations futures non corrélées et de moyenne conditionnelle nulle, les variances de sommes s’additionnent. Mais les distributions à plusieurs horizons, la moyenne, les covariances et les queues demandent de la prudence.

**Avant Python — sur le papier :** Une variance quotidienne prévue h=1 %² sur 5 jours constants donne une volatilité sur 5 jours de combien ?

**À observer lorsque le calcul s'exécute :** Ne pas confondre une **prévision de volatilité de somme** et la VaR exacte de la somme : sous dynamique GARCH, la distribution multi-horizon est plus subtile qu’une normale simple.

In [ ]:
h_futur=np.array([1.0,1.1,1.05,.95,1.2]) # variances conditionnelles futures prévues
var_cumulee=h_futur.sum()
print('Cinq variances quotidiennes prévues (%²) :',h_futur)
print(f'Variance de somme = {var_cumulee:.3f} %² ; écart-type ≈ {np.sqrt(var_cumulee):.3f} %')
print(f'Raccourci √5×1 % = {np.sqrt(5):.3f} %, exact seulement si les cinq variances valent 1 %² et covariances nulles.')

---
## IX · Valider sans regarder le futur

*Dans le cours : §§ 40–43*  

> **La question qui ouvre cette séquence**  
> Comment distinguer une prévision crédible d’un joli modèle qui ne fonctionne que sur son passé ?

**Notre itinéraire :** validation → refit licite → proxys → QLIKE → exceptions → tests de couverture et indépendance

### Le protocole chronologique : une frontière infranchissable

Nous avons choisi le modèle sur **validation**. Il est alors licite de réestimer ses paramètres sur **TRAIN+VALIDATION**, ce qui constitue désormais notre passé connu avant l'ouverture du test. Le test final reste entièrement hors estimation **et hors choix de modèle**.

Pour évaluer une prévision à la séance \(t\), nous utilisons \(\varepsilon_{t-1}\) lorsqu'il est observé, mais jamais \(\varepsilon_t\) avant d'avoir produit \(h_t\). Une prédiction fixe de plusieurs jours en avance nécessiterait au contraire de prévoir aussi les chocs futurs (nous avons étudié cette récursion).

In [ ]:
historique=marche.iloc[:fin_validation].copy()
mu_final=float(historique.rendement_pct.mean())
E=marche.rendement_pct.to_numpy()-mu_final
estimation_finale=estimer_garch(E[:fin_validation],asymmetric=(modele_choisi=='GJR'),maxiter=400)
print('Modèle retenu :',modele_choisi,'| réestimation sur train+validation :',estimation_finale['success'])
print('Paramètres :',np.round(estimation_finale['params'],5))
assert estimation_finale['success']
OM,AL,BE,GA=estimation_finale['params']
H=filtrer_variance(E,OM,AL,BE,gamma=GA,initial=np.var(E[:fin_train]))
H_const=np.full(len(E),np.var(E[:fin_validation]))
SL=slice(fin_validation,len(E))
H_test=H[SL];H_const_test=H_const[SL]
E_test=E[SL];R_test=marche.rendement_pct.to_numpy()[SL]
H_vrai=marche.variance_latente_pct2.to_numpy()[SL]  # oracle synthétique seulement
assert len(H_test)==len(test)==480 and (H_test>0).all()

### Expérience 33 — Comparer les prévisions de variance sans observer la vraie variance du marché

**Lien avec CH03 :** § 41.

**Pourquoi faisons-nous cela ?** Dans des données quotidiennes réelles, la variance conditionnelle latente n’est pas observable. Le carré des innovations est un proxy très bruité.

**Avant Python — sur le papier :** Quand un rendement observé vaut zéro, sa variance conditionnelle était-elle nécessairement zéro ?

**À observer lorsque le calcul s'exécute :** La forme QLIKE `log(h)+proxy/h` est utilisée à constante commune près. Les **classements** sont comparables, même si le score peut être négatif. Un proxy quotidien est bruité ; sa moyenne conditionnelle n’est h que lorsque le modèle de moyenne est correctement spécifié.

In [ ]:
def qlike(proxie,prevision):
 prevision=np.maximum(np.asarray(prevision,float),1e-12)
 proxie=np.maximum(np.asarray(proxie,float),0.)
 # A une constante indépendante du modèle près : log(h)+proxy/h.
 return float(np.mean(np.log(prevision)+proxie/prevision))
proxie=E_test**2
scores=pd.DataFrame([{'Modèle':'Variance constante','QLIKE (proxy ε²)':qlike(proxie,H_const_test),
                      'MSE vs variance vraie synthétique':np.mean((H_const_test-H_vrai)**2)},
                     {'Modèle':modele_choisi,'QLIKE (proxy ε²)':qlike(proxie,H_test),
                      'MSE vs variance vraie synthétique':np.mean((H_test-H_vrai)**2)}])
display(scores.round(5))
print('Le MSE versus h vraie est accessible uniquement parce que ce marché est simulé.')

### Expérience 34 — Estimer ν sans toucher aux observations de test

**Lien avec CH03 :** §§ 32, 40–42.

**Pourquoi faisons-nous cela ?** Nous comparons une innovation normale et une Student-t standardisée, avec ν appris uniquement sur TRAIN+VALIDATION.

**Avant Python — sur le papier :** Pourquoi ν>2 pour standardiser la variance de Student-t ? Que devient la loi quand ν devient très grand ?

**À observer lorsque le calcul s'exécute :** Ce calcul ajuste la forme marginale des innovations standardisées. Il ne remplace pas l’estimation conjointe complète d’un GARCH Student-t, et sa précision peut en dépendre.

In [ ]:
z_passe=E[:fin_validation]/np.sqrt(H[:fin_validation])
def nll_t_standard(nu,z):
 s=np.sqrt((nu-2.)/nu)
 return -np.sum(stats.t.logpdf(z/s,nu)-np.log(s))
nu_opt=minimize_scalar(lambda nu:nll_t_standard(nu,z_passe),bounds=(2.1,50),method='bounded')
nu_est=float(nu_opt.x)
print('ν estimé sur train+validation :',round(nu_est,3),'| succès :',nu_opt.success)
print('NLL normale :',round(-np.sum(stats.norm.logpdf(z_passe)),1),
      '| NLL Student-t standardisée :',round(nu_opt.fun,1))
assert nu_opt.success and nu_est>2

### Une VaR à 99 % est une suite de prévisions, pas un seuil unique

À chaque date de test, nous disposons de \(\mu\) appris avant le test et d'une variance \(h_t\) actualisée à partir des observations déjà connues. Pour une position de \(100\,000\) DH, nous calculons la **VaR du lendemain**, puis nous observons ce qui s'est réellement passé. Un dépassement est un jour où la perte réelle **excède** le seuil prédit : \(L_t>\mathrm{VaR}_{t}\).

In [ ]:
P=.01;CAPITAL=100_000.
s=np.sqrt((nu_est-2.)/nu_est)
qt_brut=stats.t.ppf(P,nu_est)
qt_std=s*qt_brut
q_norm=stats.norm.ppf(P)
VaR_norm=-(CAPITAL/100)*(mu_final+np.sqrt(H_test)*q_norm)
VaR_t=-(CAPITAL/100)*(mu_final+np.sqrt(H_test)*qt_std)
loss_test=-CAPITAL*R_test/100
viol_norm=loss_test>VaR_norm
viol_t=loss_test>VaR_t
# ES Student-t conditionnelle, standardisée à variance unitaire
Esz_t=-s*(nu_est+qt_brut**2)/(nu_est-1)*stats.t.pdf(qt_brut,nu_est)/P
ES_norm=(CAPITAL/100)*(-mu_final+np.sqrt(H_test)*stats.norm.pdf(q_norm)/P)
ES_t=-(CAPITAL/100)*(mu_final+np.sqrt(H_test)*Esz_t)
assert np.all(ES_t>=VaR_t) and np.all(ES_norm>=VaR_norm)
print(f'Sur {len(R_test)} séances de test, cible théorique : {len(R_test)*P:.1f} dépassements')
print('Normale :',int(viol_norm.sum()),'| Student-t :',int(viol_t.sum()),'| ν estimé :',round(nu_est,2))

### Expérience 35 — Backtest : voir les pertes et les seuils jour par jour

**Lien avec CH03 :** §§ 40–42.

**Pourquoi faisons-nous cela ?** Un total global de violations peut masquer une mauvaise localisation des épisodes de risque.

**Avant Python — sur le papier :** La VaR 99 % prévoit environ 1 % de dépassements **sous un modèle correctement calibré**, pas exactement 1 % sur chaque échantillon fini.

**À observer lorsque le calcul s'exécute :** La VaR doit avoir été prévue **avant** la perte représentée. Les points rouges ne permettent pas d’affirmer automatiquement que la loi Student-t est meilleure sur d’autres données.

In [ ]:
idx=np.arange(len(R_test)); dernier=slice(-170,None)
fig,ax=plt.subplots(figsize=(11,4))
ax.plot(idx[dernier],loss_test[dernier],color=MUTED,lw=1,label='Perte signée observée (DH)')
ax.plot(idx[dernier],VaR_norm[dernier],color=COLOR,label='VaR normale 99 %')
ax.plot(idx[dernier],VaR_t[dernier],color=ACCENT,label='VaR Student-t 99 %')
points=idx[dernier][viol_t[dernier]]
ax.scatter(points,loss_test[dernier][viol_t[dernier]],color='crimson',s=42,label='Dépassements Student-t',zorder=4)
ax.set(xlabel='Indice de séance (sur la fin du TEST)',ylabel='Perte signée / seuil (DH)',title='La validation est chronologique : un seuil par séance');ax.legend(loc='upper left',fontsize=8);plt.show()

### Deux questions différentes, deux familles de tests

**Couverture inconditionnelle (Kupiec).** Pour une VaR à 99 %, attendons-nous environ \(1\%\) de violations ? Un écart observé peut être dû au hasard : le test compare un modèle imposant \(p=0,01\) à un modèle où la fréquence observée est libre. La statistique de rapport de vraisemblance suit asymptotiquement une loi \(\chi^2_1\) sous les hypothèses usuelles.

**Indépendance (Christoffersen).** Les violations viennent-elles en grappes ? Nous comptons les transitions \(0\to0\), \(0\to1\), \(1\to0\), \(1\to1\). Si une violation rend la suivante plus probable, le modèle de risque peut ne pas réagir assez vite. La statistique d'indépendance est elle aussi comparée asymptotiquement à \(\chi^2_1\).

**Attention :** peu de dépassements à 99 % rendent ces tests peu puissants et parfois peu fiables en approximation asymptotique. On ne transforme jamais une p-value supérieure à 0,05 en preuve que le modèle est correct.

In [ ]:
def test_kupiec(exceptions, p=.01):
 x=np.asarray(exceptions,int);n=len(x);k=int(x.sum())
 phat=k/n
 ll0=xlogy(k,p)+xlogy(n-k,1-p)
 ll1=xlogy(k,phat)+xlogy(n-k,1-phat)
 LR=max(0.,float(2*(ll1-ll0)))
 return {'n':n,'violations':k,'taux':phat,'LR_UC':LR,'p_UC_asymptotique':float(stats.chi2.sf(LR,1))}

def test_independance(exceptions):
 x=np.asarray(exceptions,int);a=x[:-1];b=x[1:]
 n00=int(np.sum((a==0)&(b==0)));n01=int(np.sum((a==0)&(b==1)))
 n10=int(np.sum((a==1)&(b==0)));n11=int(np.sum((a==1)&(b==1)))
 p01=n01/(n00+n01) if (n00+n01)>0 else 0.
 p11=n11/(n10+n11) if (n10+n11)>0 else 0.
 pi=(n01+n11)/(n00+n01+n10+n11)
 def bin_ll(succ,total,prob):
  return float(xlogy(succ,prob)+xlogy(total-succ,1-prob))
 l0=bin_ll(n01+n11,len(a),pi)
 l1=bin_ll(n01,n00+n01,p01)+bin_ll(n11,n10+n11,p11)
 LR=max(0.,2*(l1-l0))
 return {'n00':n00,'n01':n01,'n10':n10,'n11':n11,
         'LR_IND':LR,'p_IND_asymptotique':float(stats.chi2.sf(LR,1))}

for nom,hits in [('Normale',viol_norm),('Student-t',viol_t)]:
 print('\n',nom)
 print('Couverture :',test_kupiec(hits,P))
 print('Indépendance :',test_independance(hits))

### Expérience 36 — Deux modèles avec même fréquence, mais des violations groupées

**Lien avec CH03 :** § 42.

**Pourquoi faisons-nous cela ?** Une fréquence globale correcte est compatible avec un régime où les violations se concentrent dans une crise.

**Avant Python — sur le papier :** Construisez deux suites de 200 jours contenant chacune 4 violations : espacées dans l’une, consécutives dans l’autre.

**À observer lorsque le calcul s'exécute :** L’expérience est volontairement artificielle. Elle illustre **pourquoi** on teste l’indépendance, pas la puissance d’un test sur quatre observations.

In [ ]:
v_espace=np.zeros(200,int);v_espace[[20,70,120,170]]=1
v_groupe=np.zeros(200,int);v_groupe[[70,71,72,73]]=1
for nom,v in [('Espacés',v_espace),('Groupés',v_groupe)]:
 print(nom,'| taux =',v.mean(),'| transitions et p-value :',test_independance(v))
fig,ax=plt.subplots(figsize=(10,2.5))
ax.scatter(np.flatnonzero(v_espace),np.ones(v_espace.sum()),label='Espacés',color=COLOR,s=48)
ax.scatter(np.flatnonzero(v_groupe),np.zeros(v_groupe.sum()),label='Groupés',color=ACCENT,s=48)
ax.set(xlabel='Séance',yticks=[0,1],yticklabels=['groupés','espacés'],title='Même 2 % de violations, localisation différente');plt.show()

### Expérience 37 — Pourquoi l’ES réclame un contrôle distinct

**Lien avec CH03 :** § 43.

**Pourquoi faisons-nous cela ?** L’ES renseigne sur la sévérité moyenne des pertes extrêmes, que la seule fréquence de violation VaR ne suffit pas à mesurer.

**Avant Python — sur le papier :** Sur les jours où la VaR est dépassée, comparez moyenne des pertes et moyenne des ES prévues ; combien de jours sont concernés ?

**À observer lorsque le calcul s'exécute :** L’ES a besoin de procédures de backtest adaptées (tests de calibration, scores conjoints VaR–ES, etc.). Comparer deux moyennes conditionnées sur les violations n’est **pas** un test formel ni une validation.

In [ ]:
pour_viol_t=loss_test[viol_t]
print('Nombre de dépassements Student-t :',len(pour_viol_t))
if len(pour_viol_t):
 print('Perte moyenne parmi les jours dépassant la VaR :',round(pour_viol_t.mean(),2),'DH')
 print('ES moyenne prédite sur ces mêmes jours :',round(ES_t[viol_t].mean(),2),'DH')
else:
 print('Aucun dépassement observé : comparaison empirique trop pauvre ici.')
print('Attention : ce simple rapprochement n’est PAS un backtest ES statistiquement valide.')

> **Arrêt de compréhension**  
> Une p-value de Kupiec supérieure à 0,05 signifie-t-elle que la VaR est correcte ?

---
## X · De la prévision à la décision — mission intégratrice

*Dans le cours : §§ 44–49*  

> **La question qui ouvre cette séquence**  
> Une direction veut engager 100 000 DH demain : quel montant de risque peut-on justifier avec les informations disponibles ?

**Notre itinéraire :** dimensionnement prudent → comparaison → note argumentée → tests d’autonomie

### Expérience 38 — Dimensionnement avec cible de volatilité et plafond de levier

**Lien avec CH03 :** § 44.

**Pourquoi faisons-nous cela ?** Une même position nominale expose davantage quand la volatilité prévue augmente. On peut réduire l’exposition, mais le risque de perte extrême ne devient pas nul.

**Avant Python — sur le papier :** Cible de volatilité 1 % par jour ; si σ prévue vaut 2 %, quel multiplicateur d’exposition pour atteindre la cible ?

**À observer lorsque le calcul s'exécute :** Cette règle de ciblage n’est pas une stratégie démontrée profitable. Les pertes, les frais, le financement, la liquidité et les contraintes réglementaires sont absents de la simulation.

In [ ]:
vol_cible=1.0
sig_pred=np.sqrt(H_test)
expo=np.clip(vol_cible/np.maximum(sig_pred,1e-8),0.,1.5)
notionnel=100_000*expo
print('Cinq premières décisions du test :')
display(pd.DataFrame({'sigma_pred_pct':sig_pred[:5],'multiplicateur':expo[:5],
                      'exposition_DH':notionnel[:5],'VaR_t_DH_unite':VaR_t[:5]}).round(2))
fig,ax=plt.subplots(figsize=(10.5,3.5));ax.plot(sig_pred,label='σ prévue (%)',color=ACCENT)
ax.plot(expo,label='Multiplicateur d’exposition',color=COLOR)
ax.set(xlabel='Séance TEST',ylabel='Valeur (unités précisées en légende)',title='Moins d’exposition lorsque la volatilité prévue monte');ax.legend();plt.show()
assert (expo>=0).all() and (expo<=1.5).all()

### La mission finale — dossier à remettre

**Votre rôle :** analyste des risques d'une société fictive. La direction envisage une exposition de référence de **100 000 DH** sur l'actif du laboratoire. Elle veut connaître la volatilité conditionnelle prévue, la VaR 99 %, l'ES 99 %, ainsi que les limites de ces chiffres.

**Règle de l'exercice :** ne pas rechercher un signal d'achat/vente. Vous devez comparer un modèle de variance constante à une dynamique GARCH et, si la validation le justifie, un GJR. Le choix de modèle se fait sur **validation**, jamais sur test. La loi Student-t est calibrée sans test futur. Les décisions sont calculées avant le rendement à évaluer.

**Votre note finale doit répondre en prose complète aux questions suivantes :**

1. **Problème et données.** Qu'est-ce qui est simulé ? Que signifie une volatilité latente ? Pourquoi ne peut-on pas l'observer sur des données réelles ?
2. **Mécanisme.** Dans notre modèle, que représentent \(\omega\), \(\alpha\), \(\beta\), et éventuellement \(\gamma\) ?
3. **Méthode.** Décrivez en dates et en effectifs les trois sous-échantillons. À quel moment a été choisi le modèle ?
4. **Qualité.** Comparez les scores de variance et les diagnostics des innovations standardisées, sans prétendre que « non-significatif » signifie « modèle vrai ».
5. **Mesures de perte.** Pour la dernière séance du test, indiquez les VaR et ES normales / Student-t, en DH, et interprétez exactement le niveau 99 %.
6. **Décision sous contraintes.** Expliquez comment une cible de volatilité et un plafond d’exposition peuvent aider à dimensionner une position, sans garantir les pertes.
7. **Limites.** Qu'est-ce qui manque avant toute décision sur un actif réel (frais, liquidité, ruptures, sauts, risque de modèle, stabilité temporelle, etc.) ?

La note sera jugée sur **la justesse des explications**, **l'absence de fuite temporelle**, **les unités**, **la prudence des conclusions** et **la qualité des contre-exemples**, pas sur la recherche d'une performance financière.

In [ ]:
bilan=pd.DataFrame({'Indicateur':['Modèle retenu','ν (Student-t)','Volatilité prévue dernière séance (%)',
                               'VaR normale dernière séance (DH)','VaR Student-t dernière séance (DH)',
                               'ES normale dernière séance (DH)','ES Student-t dernière séance (DH)',
                               'Taux de violations normal (%)','Taux de violations Student-t (%)'],
                    'Valeur':[modele_choisi,round(nu_est,3),round(float(sig_pred[-1]),3),
                              round(float(VaR_norm[-1]),2),round(float(VaR_t[-1]),2),
                              round(float(ES_norm[-1]),2),round(float(ES_t[-1]),2),
                              round(100*viol_norm.mean(),2),round(100*viol_t.mean(),2)]})
display(bilan)
print('Les valeurs sont propres au scénario synthétique et à cette réalisation pseudo-aléatoire.')

### Auto-évaluation finale — douze questions sans calculatrice

1. Pourquoi deux actifs de même espérance peuvent-ils présenter des risques très différents ?
2. Pourquoi un rendement négatif peut-il être une surprise positive ?
3. Que signifie la barre verticale dans \(\mathbb E(r_t\mid\mathcal F_{t-1})\) ?
4. Pourquoi élève-t-on les écarts à la moyenne au carré ?
5. Pourquoi une ACF faible des rendements n'exclut-elle pas une forte ACF de leurs carrés ?
6. Quelle information appartient légitimement à \(\mathcal F_{t-1}\) ?
7. Que mesure la somme \(\alpha+\beta\) dans un GARCH standard ?
8. Pourquoi une convergence d'optimiseur ne suffit-elle pas à valider un modèle ?
9. Quelle différence entre Student-t brute et Student-t standardisée ?
10. Pourquoi le nombre de violations VaR ne suffit-il pas pour juger un modèle ?
11. Pourquoi VaR 99 % n'est-elle pas une perte maximale ?
12. Pourquoi la volatilité latente utilisée dans notre étude est-elle un privilège pédagogique ?

**Consigne :** répondez avec vos mots, puis retrouvez le graphique ou le calcul du notebook qui justifie votre réponse.

## Conclusion : ce que nous avons réellement appris

Nous avons commencé avec un calcul modeste : deux actifs pouvaient avoir la même moyenne et des fluctuations très différentes. Nous avons construit la variance à partir des écarts, puis montré que **l'information disponible modifie non seulement la prévision moyenne, mais aussi sa dispersion**.

Les observations ont fait apparaître une mémoire des amplitudes, modélisée par ARCH et GARCH. Nous avons calculé les équations à la main, programmé leur récursion, puis appris des paramètres sans utiliser le futur. Nous avons testé les résidus, les queues épaisses et l'asymétrie, traduit les distributions en VaR et ES, puis examiné si les seuils résistaient aux observations ultérieures.

**Le véritable acquis n'est pas un coefficient ni une p-value : c'est une méthode.** À chaque fois, se demander **ce qui était connu**, **ce que suppose le modèle**, **ce qui a été calculé**, **ce que le test permet réellement de conclure** et **ce qui reste incertain**.

### Sources scientifiques, pour approfondir après le cours

- R. F. Engle (1982), *Autoregressive Conditional Heteroscedasticity with Estimates of the Variance of United Kingdom Inflation*, **Econometrica**, 50(4), 987–1007. DOI : [10.2307/1912773](https://doi.org/10.2307/1912773).
- T. Bollerslev (1986), *Generalized Autoregressive Conditional Heteroskedasticity*, **Journal of Econometrics**, 31(3), 307–327. DOI : [10.1016/0304-4076(86)90063-1](https://doi.org/10.1016/0304-4076(86)90063-1).
- P. F. Christoffersen (1998), *Evaluating Interval Forecasts*, **International Economic Review**, 39(4), 841–862.
- A. J. Patton (2011), *Volatility Forecast Comparison Using Imperfect Volatility Proxies*, **Journal of Econometrics**, 160(1), 246–256. DOI : [10.1016/j.jeconom.2010.03.034](https://doi.org/10.1016/j.jeconom.2010.03.034).

**Prochaine étape du cours : CH04.** Une seule volatilité ne suffit pas pour analyser un portefeuille : il faut aussi comprendre comment les actifs évoluent **ensemble** et comment les covariances modifient le risque total.